<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-10-style-tune-the-lumina-assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 10 (graded) — Style-tune the Lumina assistant
**Course 2: Generative AI and LLMs with Python — Chapter 10: Fine-tuning: LoRA / QLoRA**

**Problem brief (Leo Farkas, Lumina Health):** "The assistant's answers are correct but they
don't sound like our clinical guidelines — wrong format, wrong caveats. Prompting isn't
getting us there. Can we teach the model our style?"

**What you'll submit:** a working LoRA adapter, a before/after eval against the base model,
and a written "prompt vs RAG vs fine-tune" recommendation.

In [ ]:
!pip install -q peft trl bitsandbytes transformers accelerate datasets

## 1. Build the instruction data: Dolly-15k, restyled for Lumina's format
Lumina's house style — a direct answer, then a caveat sentence — is applied to a real Dolly
instruction subset so there's an actual learnable stylistic signal, not just topic content.
**Offline fallback:** a small hand-written set with the same style, if the download fails.

In [ ]:
import random
random.seed(0)

CAVEAT = ' Please confirm with your care team before acting on this information.'

def restyle(instruction, response):
    styled = response.strip()
    if not styled.endswith('.'):
        styled += '.'
    return f'{styled}{CAVEAT}'

def load_style_data(n=300):
    try:
        from datasets import load_dataset
        ds = load_dataset('databricks/databricks-dolly-15k', split=f'train[:{n}]')
        examples = [{'instruction': r['instruction'], 'response': restyle(r['instruction'], r['response'])}
                    for r in ds if len(r['response']) > 20]
        print(f'Loaded {len(examples)} real Dolly-15k examples, restyled.')
        return examples
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — a small hand-written set in the same style.')
        base = [
            ('What should I do if I miss a dose of medication?',
             'Take it as soon as you remember unless it is almost time for the next dose, in which case skip it.'),
            ('How much water should an adult drink daily?',
             'Roughly 2 to 3 liters per day is a common general guideline.'),
            ('What are signs of dehydration?',
             'Common signs include dark urine, dizziness, and dry mouth.'),
        ]
        return [{'instruction': i, 'response': restyle(i, r)} for i, r in base] * 30

style_data = load_style_data()
print(style_data[0])

## 2. Load the base model (4-bit if bitsandbytes is available)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'  # small enough to fine-tune on a free T4, or CPU (slower)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    if device == 'cuda':
        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
        print('Loaded in 4-bit (QLoRA-style) on GPU.')
    else:
        raise RuntimeError('no GPU')
except Exception as e:
    print(f'4-bit load skipped ({e}) — loading in full precision on {device}.')
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)

## 3. Attach a LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if hasattr(base_model, 'is_loaded_in_4bit') and base_model.is_loaded_in_4bit:
    base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # attention projections
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 4. Fine-tune with `trl`'s SFTTrainer

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def to_chat_text(example):
    messages = [{'role': 'user', 'content': example['instruction']},
                {'role': 'assistant', 'content': example['response']}]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = Dataset.from_list(style_data).map(to_chat_text)

sft_config = SFTConfig(
    output_dir='./lumina-lora',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    report_to=[],
    max_length=256,  # renamed from max_seq_length in newer trl releases
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_ds)
trainer.train()

## 5. Before/after comparison

In [ ]:
def generate_reply(model, prompt, max_new_tokens=60):
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                            return_tensors='pt').to(model.device)
    # apply_chat_template(tokenize=True) now returns a BatchEncoding, not a raw tensor
    prompt_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)

test_prompts = [
    'What are common signs of dehydration?',
    'How often should I check my blood pressure at home?',
]

for p in test_prompts:
    print(f'PROMPT: {p}')
    print(f'  Fine-tuned: {generate_reply(model, p)}')
    with model.disable_adapter():
        print(f'  Base model: {generate_reply(model, p)}')
    print()

In [ ]:
caveat_phrase = 'confirm with your care team'

def style_match_rate(model, prompts):
    return sum(caveat_phrase in generate_reply(model, p).lower() for p in prompts) / len(prompts)

with model.disable_adapter():
    base_rate = style_match_rate(base_model, test_prompts)
tuned_rate = style_match_rate(model, test_prompts)
print(f'Style-match rate (mentions the caveat) — base: {base_rate:.0%}  fine-tuned: {tuned_rate:.0%}')

## 6. Prompt vs. RAG vs. fine-tune recommendation (fill in)
Given the before/after result above, and the fact that Lumina's actual problem was format/
tone (not missing facts), was fine-tuning the right lever here — or could a well-designed
system prompt have gotten most of the way there for less cost?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 10: Fine-tuning: LoRA / QLoRA*